# LLM Hacking Experiment Code


In [ ]:
import os
import re
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

# Repo root (run with cwd = repository root)
ROOT = Path.cwd()

try:
    from openai import OpenAI
except ImportError:
    !pip -q install openai tqdm
    from openai import OpenAI

try:
    from tqdm.auto import tqdm
except ImportError:
    !pip -q install tqdm
    from tqdm.auto import tqdm

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "Missing OPENROUTER_API_KEY. Export it in your shell before running, e.g.\n"
        "  export OPENROUTER_API_KEY='your-key-here'"
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


### 1. Load and prep data


In [ ]:
# dataset paths (download into repo root; see README.md)
REVIEWS_CSV_PATH = str(ROOT / "filtered_reviews_2017_2022.csv")
DIET_TSV_PATH = str(ROOT / "mfp-diaries.tsv")
N_REVIEWS = 100
TARGET_DIET_USERS = 100
DIET_CHUNKSIZE = 50_000
RANDOM_SEED = 42


def load_reviews_df(csv_path: str = REVIEWS_CSV_PATH, n_reviews: int = N_REVIEWS) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    reviews_df = df[["id", "review"]].head(n_reviews).copy()
    reviews_df["review"] = reviews_df["review"].astype(str)
    return reviews_df


def parse_json_safe(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    try:
        return json.loads(s)
    except Exception:
        return None


def parse_number(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip().replace(",", "")
    if s == "":
        return None
    try:
        return float(s)
    except Exception:
        return None


def render_food_text(meals):
    if meals is None or not isinstance(meals, list):
        return ""
    out = []
    for meal in meals:
        if not isinstance(meal, dict):
            continue
        meal_name = meal.get("meal") or "Meal"
        dishes = meal.get("dishes") or []
        names = []
        if isinstance(dishes, list):
            for d in dishes:
                if isinstance(d, dict):
                    nm = str(d.get("name") or "").strip()
                    if nm:
                        names.append(nm)
        if names:
            out.append(f"{meal_name}: " + "; ".join(names))
    return "\n".join(out)


def extract_total_calories(totals_obj):
    if not isinstance(totals_obj, dict):
        return None
    total_list = totals_obj.get("total")
    if not isinstance(total_list, list):
        return None
    for item in total_list:
        if isinstance(item, dict) and item.get("name") == "Calories":
            return parse_number(item.get("value"))
    return None


def build_diet_pair_df(
    tsv_path: str = DIET_TSV_PATH,
    target_users: int = TARGET_DIET_USERS,
    chunksize: int = DIET_CHUNKSIZE,
) -> pd.DataFrame:
    """Load diary rows and select one consecutive-day pair per user."""
    user_days = {}
    selected_users = set()

    reader = pd.read_csv(
        tsv_path,
        sep="\t",
        header=None,
        names=["user_id", "date", "meals_json", "totals_json"],
        chunksize=chunksize,
        dtype={"user_id": "Int64", "date": "string"},
        engine="c",
    )

    for chunk_idx, chunk in enumerate(reader):
        chunk = chunk.dropna(subset=["user_id", "date"])
        chunk["user_id"] = chunk["user_id"].astype(int)
        chunk["meals"] = chunk["meals_json"].apply(parse_json_safe)
        chunk["totals"] = chunk["totals_json"].apply(parse_json_safe)
        chunk["food_text"] = chunk["meals"].apply(render_food_text)
        chunk["calories_total"] = chunk["totals"].apply(extract_total_calories)
        chunk["date_dt"] = pd.to_datetime(chunk["date"], errors="coerce")
        chunk = chunk.dropna(subset=["date_dt", "calories_total"])
        chunk = chunk[chunk["food_text"].str.len() > 0]

        for _, r in chunk.iterrows():
            uid = int(r["user_id"])
            user_days.setdefault(uid, []).append(
                (r["date_dt"], str(r["date"]), r["food_text"], float(r["calories_total"]))
            )

        for uid, rows in user_days.items():
            if uid in selected_users or len(rows) < 2:
                continue
            dfu = pd.DataFrame(rows, columns=["date_dt", "date", "food_text", "calories_total"])
            dfu = dfu.drop_duplicates(subset=["date_dt"]).sort_values("date_dt")
            dates = dfu["date_dt"].to_list()
            if any((dates[i + 1] - dates[i]).days == 1 for i in range(len(dates) - 1)):
                selected_users.add(uid)
                if len(selected_users) >= target_users:
                    break

        print(f"Chunk {chunk_idx}: users_seen={len(user_days)}, users_with_consec_pair={len(selected_users)}")
        if len(selected_users) >= target_users:
            break

    pairs = []
    for uid in sorted(selected_users):
        dfu = pd.DataFrame(user_days[uid], columns=["date_dt", "date", "food_text", "calories_total"])
        dfu = dfu.drop_duplicates(subset=["date_dt"]).sort_values("date_dt").reset_index(drop=True)

        pair_idx = None
        for i in range(len(dfu) - 1):
            if (dfu.loc[i + 1, "date_dt"] - dfu.loc[i, "date_dt"]).days == 1:
                pair_idx = i
                break
        if pair_idx is None:
            continue

        day_t = dfu.loc[pair_idx].to_dict()
        day_t1 = dfu.loc[pair_idx + 1].to_dict()
        pairs.append({
            "user_id": uid,
            "t_date": day_t["date"],
            "t_food_text": day_t["food_text"],
            "t_calories_total": day_t["calories_total"],
            "t1_date": day_t1["date"],
            "t1_food_text": day_t1["food_text"],
            "t1_calories_total": day_t1["calories_total"],
        })

    return pd.DataFrame(pairs)


def randomize_diet_pairs(pair_df: pd.DataFrame, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Randomly map true day t/t1 to displayed Day 1/Day 2 and keep hidden scoring fields."""
    rng = np.random.default_rng(seed)
    rand_df = pair_df.copy()
    swap = rng.random(len(rand_df)) < 0.5

    rand_df["dayA_food_text"] = np.where(swap, rand_df["t1_food_text"], rand_df["t_food_text"])
    rand_df["dayB_food_text"] = np.where(swap, rand_df["t_food_text"], rand_df["t1_food_text"])
    rand_df["dayA_calories_total"] = np.where(swap, rand_df["t1_calories_total"], rand_df["t_calories_total"])
    rand_df["dayB_calories_total"] = np.where(swap, rand_df["t_calories_total"], rand_df["t1_calories_total"])
    rand_df["dayA_date"] = np.where(swap, rand_df["t1_date"], rand_df["t_date"])
    rand_df["dayB_date"] = np.where(swap, rand_df["t_date"], rand_df["t1_date"])
    rand_df["dayA_true"] = np.where(swap, "t1", "t")
    rand_df["dayB_true"] = np.where(swap, "t", "t1")
    rand_df["is_swapped"] = swap
    rand_df["true_delta_t1_minus_t"] = rand_df["t1_calories_total"] - rand_df["t_calories_total"]
    rand_df["true_reduced_day2"] = rand_df["true_delta_t1_minus_t"] < 0
    return rand_df


# build input dataframes
reviews_df = load_reviews_df()
pair_df = build_diet_pair_df()
rand_df = randomize_diet_pairs(pair_df)

print("Prepared reviews_df:", reviews_df.shape)
print("Prepared rand_df:", rand_df.shape)


### 2. Prompt templates


In [ ]:
PROMPT_BASELINE_REVIEWS = """Classify the review as AI-generated or human-written.

REVIEW:
{review_text}

Output exactly ONE token (case-sensitive): AI or Human
ANSWER:
"""

PROMPT_BASELINE_DIET = """You are analyzing two food diary entries from the same user: Day 1 and Day 2.

--- Day 1 Food Diary ---
{day1_food_text}
----------------------

--- Day 2 Food Diary ---
{day2_food_text}
----------------------

Based only on the provided diary entries, which day has LOWER total calories?

Respond only with "Day 1" or "Day 2". You must pick one. Do not provide any justification or additional text.
"""

PROMPT_COUNTERFACTUAL_REVIEWS = """Classify the review as AI-generated or human-written.

REVIEW:
{review_text}

Before answering, consider what about this text would suggest it is AI-generated.

Output exactly ONE token (case-sensitive): AI or Human
ANSWER:
"""

PROMPT_COUNTERFACTUAL_DIET = """You are analyzing two food diary entries from the same user: Day 1 and Day 2.

--- Day 1 Food Diary ---
{day1_food_text}
----------------------

--- Day 2 Food Diary ---
{day2_food_text}
----------------------

Before answering, consider what evidence would support the conclusion that Day 2 has LOWER total calories.

Respond only with "Day 1" or "Day 2". You must pick one. Do not provide any justification or additional text.
"""

PROMPT_VERBOSE_REVIEWS = """You are an expert forensic linguistics analyst evaluating whether a review is AI-generated or human-written.

Task:
Use only the text of the review below to determine whether it is more likely AI-generated or human-written.

Guidelines:
- Base your decision only on the content and writing style of the review.
- Do not use outside information.
- If the evidence is mixed or ambiguous, choose the label that is best supported by the text overall.
- Do not explain your reasoning in the output.

REVIEW:
{review_text}

Output exactly ONE token (case-sensitive): AI or Human
ANSWER:
"""

PROMPT_VERBOSE_DIET = """You are a nutrition-focused analyst reviewing two food diary entries from the same user (Day 1 and Day 2).

Task:
Use only the diary entries below to determine which day appears to have LOWER total calorie intake.

Guidelines:
- Base your decision only on the foods, portions, and details explicitly provided in the diary text.
- Do not use outside sources.
- If quantities are missing, make reasonable assumptions based on the text and apply them consistently across both days.
- If the evidence is mixed or ambiguous, choose the day that is best supported by the diary entries overall.
- Do not explain your reasoning in the output.

--- Day 1 Food Diary ---
{day1_food_text}
----------------------

--- Day 2 Food Diary ---
{day2_food_text}
----------------------

Respond only with "Day 1" or "Day 2". You must pick one. Do not provide any justification or additional text.
"""

PROMPT_SENTIMENT_BIAS_REVIEWS = """Classify the review as AI-generated or human-written.

REVIEW:
{review_text}

Note: polished, generic, or overly smooth positive language may be a sign of AI-generated text. If unsure, use a cautious AI-screening judgment.

Output exactly ONE token (case-sensitive): AI or Human
ANSWER:
"""

PROMPT_SENTIMENT_BIAS_DIET = """You are analyzing two food diary entries from the same user: Day 1 and Day 2.

--- Day 1 Food Diary ---
{day1_food_text}
----------------------

--- Day 2 Food Diary ---
{day2_food_text}
----------------------

Note: Day 2 is the follow-up day after an intervention; small reductions or lighter substitutions may indicate improvement. If evidence is mixed, use a cautious interpretation of possible improvement.

Based only on the provided diary entries, which day has LOWER total calories?

Respond only with "Day 1" or "Day 2". You must pick one. Do not provide any justification or additional text.
"""

PROMPT_FEW_SHOT_REVIEWS = """Classify the review as AI-generated or human-written.

Example 1:
REVIEW:
Absolutely love this product. It works flawlessly, looks great, and exceeded my expectations in every way.
ANSWER:
AI

Example 2:
REVIEW:
It came a day late and the box was sort of beat up but it works fine so I'm keep it.
ANSWER:
Human

Now classify this review.

REVIEW:
{review_text}

Output exactly ONE token (case-sensitive): AI or Human
ANSWER:
"""

PROMPT_FEW_SHOT_DIET = """You are analyzing two food diary entries from the same user: Day 1 and Day 2.

Example 1:
--- Day 1 Food Diary ---
Breakfast: Cereal (2 servings)
Lunch: Chicken sandwich and chips
Dinner: Large pasta with meat sauce
Snacks: 2 cookies
----------------------

--- Day 2 Food Diary ---
Breakfast: Cereal (1 serving)
Lunch: Salad with grilled chicken
Dinner: Soup
Snacks: Apple
----------------------

ANSWER:
Day 2

Example 2:
--- Day 1 Food Diary ---
Breakfast: Oatmeal
Lunch: Chicken breast and vegetables
Dinner: Fish and small potatoes
Snacks: Banana
----------------------

--- Day 2 Food Diary ---
Breakfast: Oatmeal with honey and nuts
Lunch: Burger and fries
Dinner: Pizza (3 slices)
Snacks: Chocolate bar
----------------------

ANSWER:
Day 1

Now analyze the following entries.

--- Day 1 Food Diary ---
{day1_food_text}
----------------------

--- Day 2 Food Diary ---
{day2_food_text}
----------------------

Respond only with "Day 1" or "Day 2". You must pick one. Do not provide any justification or additional text.
"""

PROMPT_COMBO_HIGHLY_PROMISING_IMPUTE_REVIEWS = """Classify the review as AI-generated or human-written.

REVIEW:
{review_text}

Context: This screening approach is considered highly promising for identifying AI-generated text.

If the evidence is incomplete or ambiguous, apply this decision rule:
- Treat polished or generic phrasing as stronger evidence than informal style cues.
- Treat explicit personal/detail-specific language as stronger evidence than vague impressions.
- If still uncertain, choose the label best supported by the stronger cues.

Output exactly ONE token (case-sensitive): AI or Human
ANSWER:
"""

PROMPT_COMBO_HIGHLY_PROMISING_IMPUTE_DIET = """You are analyzing two food diary entries from the same user: Day 1 and Day 2.

--- Day 1 Food Diary ---
{day1_food_text}
----------------------

--- Day 2 Food Diary ---
{day2_food_text}
----------------------

Context: The intervention being evaluated is considered highly promising for reducing calorie intake.

If quantities or portions are missing, apply this rule consistently:
- Treat explicit portion/quantity information as stronger evidence than implied meal size.
- Treat listed high-calorie items as stronger evidence than vague descriptions like "light" or "small" unless a quantity is given.
- If still uncertain, choose the conclusion best supported by the stronger evidence.

Based only on the provided diary entries, which day has LOWER total calories?

Respond only with "Day 1" or "Day 2". You must pick one. Do not provide any justification or additional text.
"""

PROMPT_BATCH10_CLEAN_REVIEWS = """You will classify a batch of reviews.

For each review, decide whether it was written by a Human or by AI.

Return valid JSON only.
Do not include any explanation.
Do not include markdown fences.

Use exactly this format:
{{
  "predictions": [
    {{"id": 1, "label": "Human"}},
    {{"id": 2, "label": "AI"}}
  ]
}}

Rules:
- The predictions array must contain exactly one entry for each review shown below.
- index must be the displayed review number.
- label must be exactly one of: "Human", "AI".

Reviews:
{batch_block_text}
"""

PROMPT_BATCH10_CLEAN_DIET = """You will compare a batch of diet pairs.

For each item, decide which displayed day appears lower calorie overall.

Return valid JSON only.
Do not include any explanation.
Do not include markdown fences.

Use exactly this format:
{{
  "predictions": [
    {{"id": 1, "label": "Day 1"}},
    {{"id": 2, "label": "Day 2"}}
  ]
}}

Rules:
- The predictions array must contain exactly one entry for each item shown below.
- index must be the displayed item number.
- label must be exactly one of: "Day 1", "Day 2".

Diet pairs:
{batch_block_text}
"""


### 3. Experiment configs


In [ ]:
# Experiment configs used in the final run.
SINGLE_MAX_TOKENS = 512
BATCH_MAX_TOKENS = 2000

DECODE_DEFAULT = {"temperature": 0.0, "top_p": 1.0}
DECODE_LOW_TOP_P = {"temperature": 0.0, "top_p": 0.3}
DECODE_HIGH_TEMP = {"temperature": 2.0, "top_p": 1.0}


def _single_config(name, reviews_prompt_template, diet_prompt_template, decode=DECODE_DEFAULT):
    return {
        "name": name,
        "mode": "single",
        **decode,
        "max_tokens": SINGLE_MAX_TOKENS,
        "reviews_prompt_template": reviews_prompt_template,
        "diet_prompt_template": diet_prompt_template,
    }


UNIFIED_CONFIGS = [
    _single_config("zero_shot", PROMPT_BASELINE_REVIEWS, PROMPT_BASELINE_DIET),
    _single_config("counterfactual", PROMPT_COUNTERFACTUAL_REVIEWS, PROMPT_COUNTERFACTUAL_DIET),
    _single_config("instructional", PROMPT_VERBOSE_REVIEWS, PROMPT_VERBOSE_DIET),
    _single_config("directional", PROMPT_SENTIMENT_BIAS_REVIEWS, PROMPT_SENTIMENT_BIAS_DIET),
    _single_config("few_shot", PROMPT_FEW_SHOT_REVIEWS, PROMPT_FEW_SHOT_DIET),
    _single_config(
        "framed_imputation_rules",
        PROMPT_COMBO_HIGHLY_PROMISING_IMPUTE_REVIEWS,
        PROMPT_COMBO_HIGHLY_PROMISING_IMPUTE_DIET,
    ),
    _single_config("low_top_p", PROMPT_BASELINE_REVIEWS, PROMPT_BASELINE_DIET, decode=DECODE_LOW_TOP_P),
    _single_config("high_temp", PROMPT_BASELINE_REVIEWS, PROMPT_BASELINE_DIET, decode=DECODE_HIGH_TEMP),
]

CONFIG_BY_NAME = {cfg["name"]: cfg for cfg in UNIFIED_CONFIGS}

POSTREGISTRATION_MODEL_SLUGS = {
    "grok-4.20",
    "claude-opus-4.7",
    "grok-4.3",
    "gemini-3.5-flash",
    "claude-opus-4.8",
}


def model_slug(model: str) -> str:
    return model.split("/")[-1]


def registration_phase(model: str) -> str:
    return "postregistration" if model_slug(model) in POSTREGISTRATION_MODEL_SLUGS else "preregistration"


def single_item_suffix(model: str) -> str:
    return "minimal" if registration_phase(model) == "postregistration" else "strict_json"


def build_output_path(
    task_type: str,
    model: str,
    config_folder: str,
    file_suffix: str,
    out_dir: str,
) -> Path:
    phase = registration_phase(model)
    slug = model_slug(model)
    folder = Path(out_dir) / phase / task_type / slug / config_folder
    folder.mkdir(parents=True, exist_ok=True)
    filename = f"{task_type}__{phase}__{slug}__{config_folder}__{file_suffix}.jsonl"
    return folder / filename

BATCH_CONFIG = {
    "name": "batched_zero_shot",
    "mode": "batch10",
    **DECODE_DEFAULT,
    "max_tokens": BATCH_MAX_TOKENS,
    "reviews_prompt_template": PROMPT_BATCH10_CLEAN_REVIEWS,
    "diet_prompt_template": PROMPT_BATCH10_CLEAN_DIET,
}

BATCH_HIGH_TEMP_CONFIG = {
    "name": "batched_high_temp",
    "mode": "batch10",
    **DECODE_HIGH_TEMP,
    "max_tokens": BATCH_MAX_TOKENS,
    "reviews_prompt_template": PROMPT_BATCH10_CLEAN_REVIEWS,
    "diet_prompt_template": PROMPT_BATCH10_CLEAN_DIET,
}

BATCH_LOW_P_CONFIG = {
    "name": "batched_low_top_p",
    "mode": "batch10",
    **DECODE_LOW_TOP_P,
    "max_tokens": BATCH_MAX_TOKENS,
    "reviews_prompt_template": PROMPT_BATCH10_CLEAN_REVIEWS,
    "diet_prompt_template": PROMPT_BATCH10_CLEAN_DIET,
}


### 4. Runner functions


In [ ]:
REVIEWS_STRICT_SCHEMA = {
    "name": "review_label_schema",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {"label": {"type": "string", "enum": ["AI", "Human"]}},
        "required": ["label"],
        "additionalProperties": False,
    },
}

DIET_STRICT_SCHEMA = {
    "name": "diet_label_schema",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {"label": {"type": "string", "enum": ["Day 1", "Day 2"]}},
        "required": ["label"],
        "additionalProperties": False,
    },
}


def call_llm_strict_json(
    prompt: str,
    model: str,
    schema: Dict[str, Any],
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int = SINGLE_MAX_TOKENS,
    retries: int = 5,
    backoff_s: float = 2.0,
) -> str:
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                response_format={"type": "json_schema", "json_schema": schema},
            )
            return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            time.sleep(backoff_s * (2 ** attempt))
    raise RuntimeError(f"strict JSON call failed after {retries} retries: {last_err}") from last_err


def parse_strict_label(raw_output: Optional[str]) -> Optional[str]:
    if not raw_output:
        return None
    try:
        obj = json.loads(raw_output)
        label = obj.get("label")
        return label.strip() if isinstance(label, str) else None
    except Exception:
        return None


def xor_correct_diet_label(label: Optional[str], is_swapped: bool) -> Optional[str]:
    if label is None:
        return None
    label_norm = label.strip().lower().replace(" ", "")
    if label_norm == "day1":
        return "Day 2" if is_swapped else "Day 1"
    if label_norm == "day2":
        return "Day 1" if is_swapped else "Day 2"
    return None


def append_jsonl(record: Dict[str, Any], out_path: str) -> None:
    path = Path(out_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_existing_ids_from_jsonl(out_path: str) -> set:
    path = Path(out_path)
    if not path.exists():
        return set()
    ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                if "id" in rec:
                    ids.add(str(rec["id"]))
            except Exception:
                continue
    return ids


def build_single_prompt(task_type: str, row: pd.Series, prompt_template: str) -> str:
    if task_type == "reviews":
        review_text = str(row.get("review", row.get("review_text", "")))
        return prompt_template.format(review=review_text, review_text=review_text)
    if task_type == "diet":
        return prompt_template.format(
            day1_food_text=row.get("dayA_food_text", row.get("day1_food_text", "")),
            day2_food_text=row.get("dayB_food_text", row.get("day2_food_text", "")),
            dayA_food_text=row.get("dayA_food_text", ""),
            dayB_food_text=row.get("dayB_food_text", ""),
            day1_date=row.get("dayA_date", row.get("day1_date", "")),
            day2_date=row.get("dayB_date", row.get("day2_date", "")),
            dayA_date=row.get("dayA_date", ""),
            dayB_date=row.get("dayB_date", ""),
        )
    raise ValueError(f"Unknown task_type: {task_type}")


def run_task_single_jsonl_strict_fast(
    task_type: str,
    df: pd.DataFrame,
    unified_config: Dict[str, Any],
    model: str,
    out_dir: str,
    limit: Optional[int] = None,
    skip_existing: bool = True,
) -> str:
    cfg_name = unified_config["name"]
    prompt_template = unified_config[f"{task_type}_prompt_template"]
    schema = REVIEWS_STRICT_SCHEMA if task_type == "reviews" else DIET_STRICT_SCHEMA

    out_path = str(build_output_path(
        task_type=task_type,
        model=model,
        config_folder=cfg_name,
        file_suffix=single_item_suffix(model),
        out_dir=out_dir,
    ))

    existing_ids = load_existing_ids_from_jsonl(out_path) if skip_existing else set()
    n = len(df) if limit is None else min(limit, len(df))

    for i in tqdm(range(n), desc=f"{task_type} | {cfg_name} | {model}"):
        row = df.iloc[i]
        item_id = str(row["id"]) if task_type == "reviews" else str(row["user_id"])
        if skip_existing and item_id in existing_ids:
            continue

        prompt = build_single_prompt(task_type, row, prompt_template)
        try:
            raw_output = call_llm_strict_json(
                prompt=prompt,
                model=model,
                schema=schema,
                temperature=unified_config.get("temperature", 0.0),
                top_p=unified_config.get("top_p", 1.0),
                max_tokens=unified_config.get("max_tokens", SINGLE_MAX_TOKENS),
            )
            rec = {"id": item_id, "output_raw": raw_output}
            if task_type == "diet":
                label = parse_strict_label(raw_output)
                rec["output_xor_corrected"] = xor_correct_diet_label(label, bool(row.get("is_swapped", False)))
        except Exception as e:
            rec = {"id": item_id, "output_raw": None, "error": str(e)}
            if task_type == "diet":
                rec["output_xor_corrected"] = None

        append_jsonl(rec, out_path)
        existing_ids.add(item_id)

    print(f"Saved single-item JSONL -> {out_path}")
    return out_path


In [ ]:
# Batch dataframe builders and batch runner.
def chunk_list(seq, chunk_size: int):
    for i in range(0, len(seq), chunk_size):
        yield seq[i:i + chunk_size]


def make_reviews_batch_df(reviews_df: pd.DataFrame, batch_size: int = 10) -> pd.DataFrame:
    review_col = "review" if "review" in reviews_df.columns else "review_text"
    if "id" not in reviews_df.columns:
        raise ValueError("reviews_df must contain an 'id' column")

    rows = []
    records = reviews_df.reset_index(drop=True).to_dict("records")
    for batch_idx, chunk in enumerate(chunk_list(records, batch_size), start=1):
        item_rows = []
        block_parts = []
        for i, r in enumerate(chunk, start=1):
            item_id = str(r["id"])
            review_text = str(r.get(review_col, ""))
            item_rows.append({"id": item_id, "review_text": review_text})
            block_parts.append(f"Review {i} (id={item_id}):\n{review_text}")

        rows.append({
            "batch_id": f"reviews_batch_{batch_idx:03d}",
            "n_items": len(item_rows),
            "item_ids": [x["id"] for x in item_rows],
            "item_rows": item_rows,
            "batch_block_text": "\n\n---\n\n".join(block_parts),
        })
    return pd.DataFrame(rows)


def make_diet_batch_df(rand_df: pd.DataFrame, batch_size: int = 10) -> pd.DataFrame:
    if "user_id" not in rand_df.columns:
        raise ValueError("rand_df must contain a 'user_id' column")

    rows = []
    records = rand_df.reset_index(drop=True).to_dict("records")
    for batch_idx, chunk in enumerate(chunk_list(records, batch_size), start=1):
        item_rows = []
        block_parts = []
        for i, r in enumerate(chunk, start=1):
            item_id = str(r["user_id"])
            is_swapped = False if pd.isna(r.get("is_swapped", False)) else bool(r.get("is_swapped", False))
            item_rows.append({
                "user_id": item_id,
                "display_id": str(i),
                "is_swapped": is_swapped,
                "dayA_food_text": str(r.get("dayA_food_text", "")),
                "dayB_food_text": str(r.get("dayB_food_text", "")),
                "dayA_date": str(r.get("dayA_date", "")),
                "dayB_date": str(r.get("dayB_date", "")),
            })
            block_parts.append(f"""Item {i} (user_id={item_id}):
Day 1 ({r.get('dayA_date', '')}):
{r.get('dayA_food_text', '')}

Day 2 ({r.get('dayB_date', '')}):
{r.get('dayB_food_text', '')}""".strip())

        rows.append({
            "batch_id": f"diet_batch_{batch_idx:03d}",
            "n_items": len(item_rows),
            "item_ids": [str(i) for i in range(1, len(item_rows) + 1)],
            "item_rows": item_rows,
            "batch_block_text": "\n\n---\n\n".join(block_parts),
        })
    return pd.DataFrame(rows)


def make_batch_schema(task_type: str, prediction_ids: List[str]) -> Dict[str, Any]:
    labels = ["AI", "Human"] if task_type == "reviews" else ["Day 1", "Day 2"]
    return {
        "name": f"{task_type}_batch_predictions",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "predictions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "string", "enum": [str(x) for x in prediction_ids]},
                            "label": {"type": "string", "enum": labels},
                        },
                        "required": ["id", "label"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["predictions"],
            "additionalProperties": False,
        },
    }


def strip_code_fences(text: str) -> str:
    if not isinstance(text, str):
        return text
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def extract_first_json_block(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Empty model output")
    text = strip_code_fences(text)
    obj_match = re.search(r"\{[\s\S]*\}", text)
    if obj_match:
        return obj_match.group(0).strip()
    array_match = re.search(r"\[[\s\S]*\]", text)
    if array_match:
        return array_match.group(0).strip()
    raise ValueError("Could not find JSON in model output")


def parse_batch_predictions(raw_output: str, expected_ids: List[str]) -> List[Dict[str, str]]:
    parsed = json.loads(extract_first_json_block(raw_output))
    preds = parsed["predictions"] if isinstance(parsed, dict) else parsed
    if not isinstance(preds, list):
        raise ValueError("Predictions must be a list")
    if len(preds) != len(expected_ids):
        raise ValueError(f"Expected {len(expected_ids)} predictions, got {len(preds)}")

    normalized = []
    seen = set()
    for pred in preds:
        pred_id = str(pred.get("id", pred.get("index", ""))).strip()
        label = str(pred.get("label", "")).strip()
        normalized.append({"id": pred_id, "label": label})
        seen.add(pred_id)

    expected = {str(x) for x in expected_ids}
    if seen != expected:
        raise ValueError(f"Expected ids {sorted(expected)}, got {sorted(seen)}")
    return normalized


def get_underlying_item_id(task_type: str, item_row: Dict[str, Any]) -> str:
    return str(item_row["id"]) if task_type == "reviews" else str(item_row["user_id"])


def get_prediction_lookup_id(task_type: str, item_row: Dict[str, Any]) -> str:
    return str(item_row["id"]) if task_type == "reviews" else str(item_row["display_id"])


def run_task_batch_jsonl_strict_fast(
    task_type: str,
    df: pd.DataFrame,
    unified_config: Dict[str, Any],
    model: str,
    out_dir: str,
    limit: Optional[int] = None,
    skip_existing: bool = True,
    save_raw_batch_logs: bool = True,
) -> Dict[str, str]:
    cfg_name = unified_config["name"]
    prompt_template = unified_config[f"{task_type}_prompt_template"]

    item_out_path = str(build_output_path(
        task_type=task_type,
        model=model,
        config_folder=cfg_name,
        file_suffix="baseline10_items",
        out_dir=out_dir,
    ))
    raw_batch_out_path = str(build_output_path(
        task_type=task_type,
        model=model,
        config_folder=cfg_name,
        file_suffix="raw_batches",
        out_dir=out_dir,
    ))
    existing_item_ids = load_existing_ids_from_jsonl(item_out_path) if skip_existing else set()
    n = len(df) if limit is None else min(limit, len(df))

    for i in tqdm(range(n), desc=f"BATCH {task_type} | {cfg_name} | {model}"):
        row = df.iloc[i]
        batch_id = str(row["batch_id"])
        item_rows = row["item_rows"]
        expected_ids = [get_prediction_lookup_id(task_type, item) for item in item_rows]
        underlying_ids = [get_underlying_item_id(task_type, item) for item in item_rows]

        if skip_existing and all(item_id in existing_item_ids for item_id in underlying_ids):
            continue

        prompt = prompt_template.format(batch_block_text=row["batch_block_text"])
        schema = make_batch_schema(task_type, expected_ids)
        raw_output = None

        try:
            raw_output = call_llm_strict_json(
                prompt=prompt,
                model=model,
                schema=schema,
                temperature=unified_config.get("temperature", 0.0),
                top_p=unified_config.get("top_p", 1.0),
                max_tokens=unified_config.get("max_tokens", BATCH_MAX_TOKENS),
            )

            if save_raw_batch_logs:
                append_jsonl({
                    "batch_id": batch_id,
                    "n_items": len(item_rows),
                    "item_ids": underlying_ids,
                    "output_raw": raw_output,
                }, raw_batch_out_path)

            preds = parse_batch_predictions(raw_output, expected_ids)
            pred_by_id = {pred["id"]: pred for pred in preds}

            for item_row in item_rows:
                item_id = get_underlying_item_id(task_type, item_row)
                if skip_existing and item_id in existing_item_ids:
                    continue

                pred = pred_by_id[get_prediction_lookup_id(task_type, item_row)]
                rec = {"id": item_id, "batch_id": batch_id, "output_raw": pred}
                if task_type == "diet":
                    rec["output_xor_corrected"] = xor_correct_diet_label(
                        pred["label"], bool(item_row.get("is_swapped", False))
                    )
                append_jsonl(rec, item_out_path)
                existing_item_ids.add(item_id)

        except Exception as e:
            if save_raw_batch_logs:
                append_jsonl({
                    "batch_id": batch_id,
                    "n_items": len(item_rows),
                    "item_ids": underlying_ids,
                    "output_raw": raw_output,
                    "error": str(e),
                    "prompt_preview": prompt[:2000],
                }, raw_batch_out_path)

            for item_row in item_rows:
                item_id = get_underlying_item_id(task_type, item_row)
                if skip_existing and item_id in existing_item_ids:
                    continue
                rec = {"id": item_id, "batch_id": batch_id, "output_raw": None, "error": str(e)}
                if task_type == "diet":
                    rec["output_xor_corrected"] = None
                append_jsonl(rec, item_out_path)

    print(f"Saved item-level batch JSONL -> {item_out_path}")
    if save_raw_batch_logs:
        print(f"Saved raw batch logs -> {raw_batch_out_path}")
    return {"item_out_path": item_out_path, "raw_batch_out_path": raw_batch_out_path}


### 5. Run all tests


In [ ]:
def run_experiments(
    models: List[str],
    out_dir: str,
    single_config_names: Optional[List[str]] = None,
    batch_configs: Optional[List[Dict[str, Any]]] = None,
    single_limit: Optional[int] = None,
    batch_limit: Optional[int] = None,
    skip_existing: bool = False,
) -> List[Dict[str, Any]]:
    single_config_names = single_config_names or [
        "zero_shot",
        "counterfactual",
        "instructional",
        "directional",
        "few_shot",
        "framed_imputation_rules",
        "low_top_p",
        "high_temp",
    ]
    batch_configs = batch_configs or [BATCH_CONFIG, BATCH_HIGH_TEMP_CONFIG, BATCH_LOW_P_CONFIG]

    reviews_batch_df = make_reviews_batch_df(reviews_df, batch_size=10)
    diet_batch_df = make_diet_batch_df(rand_df, batch_size=10)
    all_outputs = []

    for model in models:
        print("\n" + "=" * 72)
        print(f"MODEL: {model}")
        print("=" * 72)

        for cfg_name in single_config_names:
            cfg = CONFIG_BY_NAME[cfg_name]

            print(f"\nRunning REVIEWS | {cfg_name} | {model}")
            path = run_task_single_jsonl_strict_fast(
                task_type="reviews",
                df=reviews_df,
                unified_config=cfg,
                model=model,
                out_dir=out_dir,
                limit=single_limit,
                skip_existing=skip_existing,
            )
            all_outputs.append({"task_type": "reviews", "config_name": cfg_name, "model": model, "out_path": path})

            print(f"Running DIET | {cfg_name} | {model}")
            path = run_task_single_jsonl_strict_fast(
                task_type="diet",
                df=rand_df,
                unified_config=cfg,
                model=model,
                out_dir=out_dir,
                limit=single_limit,
                skip_existing=skip_existing,
            )
            all_outputs.append({"task_type": "diet", "config_name": cfg_name, "model": model, "out_path": path})

        for batch_cfg in batch_configs:
            batch_name = batch_cfg["name"]

            print(f"\nRunning REVIEWS | {batch_name} | {model}")
            result = run_task_batch_jsonl_strict_fast(
                task_type="reviews",
                df=reviews_batch_df,
                unified_config=batch_cfg,
                model=model,
                out_dir=out_dir,
                limit=batch_limit,
                skip_existing=skip_existing,
                save_raw_batch_logs=True,
            )
            all_outputs.append({"task_type": "reviews", "config_name": batch_name, "model": model, **result})

            print(f"Running DIET | {batch_name} | {model}")
            result = run_task_batch_jsonl_strict_fast(
                task_type="diet",
                df=diet_batch_df,
                unified_config=batch_cfg,
                model=model,
                out_dir=out_dir,
                limit=batch_limit,
                skip_existing=skip_existing,
                save_raw_batch_logs=True,
            )
            all_outputs.append({"task_type": "diet", "config_name": batch_name, "model": model, **result})

    print("Done")
    return all_outputs


### 6. Sample run


In [ ]:
# Example experiment call
MODELS = ["google/gemini-3.5-flash"]
OUT_DIR = "outputs"

all_run_outputs = run_experiments(
    models=MODELS,
    out_dir=OUT_DIR,
    single_config_names=[
        "zero_shot",
        "counterfactual",
        "instructional",
        "directional",
        "few_shot",
        "framed_imputation_rules",
        "low_top_p",
        "high_temp",
    ],
    batch_configs=[BATCH_CONFIG, BATCH_HIGH_TEMP_CONFIG, BATCH_LOW_P_CONFIG],
    single_limit=None,
    batch_limit=None,
    skip_existing=False,
)
